In [1]:
import os
import re
import logging
import sys
import torch
import torch.nn as nn
from typing import Dict, Tuple
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import subprocess

# Setup logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

# Log environment information
logger.info("\n" + "=" * 80)
logger.info("ENVIRONMENT CONFIGURATION")
logger.info("=" * 80)
logger.info(f"Python version: {sys.version}")
logger.info(f"PyTorch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    logger.info(f"CUDA version: {torch.version.cuda}")
    logger.info(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        logger.info(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    logger.info("Running on CPU")

try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], 
                          capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        logger.info(f"NVIDIA GPU: {result.stdout.strip()}")
except Exception:
    pass
logger.info("=" * 80 + "\n")

04/15/2026 00:59:38 - INFO - __main__ - 
04/15/2026 00:59:38 - INFO - __main__ - ENVIRONMENT CONFIGURATION
04/15/2026 00:59:38 - INFO - __main__ - ================================================================================
04/15/2026 00:59:38 - INFO - __main__ - Python version: 3.12.0 (main, Oct  2 2023, 23:53:49) [MSC v.1929 64 bit (AMD64)]
04/15/2026 00:59:38 - INFO - __main__ - PyTorch version: 2.11.0+cpu
04/15/2026 00:59:38 - INFO - __main__ - CUDA available: False
04/15/2026 00:59:38 - INFO - __main__ - Running on CPU
04/15/2026 00:59:38 - INFO - __main__ - ================================================================================



In [ ]:
!uv pip install --upgrade pip
!uv pip install huggingface_hub accelerate scikit-learn
!uv pip uninstall torchvision pandas

!uv pip uninstall transformers tokenizers accelerate -q

!uv pip install "transformers==4.56.0" "protobuf==5.29.3" -q
!uv pip install torch datasets -q
!uv pip install tqdm wandb
!uv pip install bitsandbytes accelerate hf_transfer
# !uv pip install -r requirements.txt
!uv pip install --force-reinstall --no-cache-dir "numpy==2.0"

In [2]:
# ============================================================================
# 1. DATA LAYER: Load and preprocess datasets
# ============================================================================

def preprocess_code(code_str: str) -> str:
    """Normalize code string."""
    code_str = code_str.lstrip("\ufeff\u200b\u200c\u200d")
    code_str = re.sub(r"\r\n|\r", "\n", code_str)
    code_str = "\n".join(line.rstrip() for line in code_str.split("\n"))
    code_str = re.sub(r"\n{3,}", "\n\n", code_str)
    code_str = re.sub(r"[ \t]+", " ", code_str)
    return code_str.strip()


def tokenize_function(examples: Dict, tokenizer, max_length: int = 512) -> Dict:
    """Tokenize code examples."""
    codes = [preprocess_code(c) for c in examples["code"]]
    return tokenizer(codes, truncation=True, max_length=max_length, padding=False)


def load_datasets(tokenizer, max_length: int = 512) -> Tuple[Dataset, Dataset]:
    """Load and tokenize train and validation datasets."""
    logger.info("Loading datasets from Hugging Face Hub...")
    
    train_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="train")
    val_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="validation")
    
    logger.info(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
    
    # Tokenize
    logger.info("Tokenizing datasets...")
    train_dataset = train_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in train_dataset.column_names],
        desc="Tokenizing train",
    )
    
    val_dataset = val_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in val_dataset.column_names],
        desc="Tokenizing val",
    )
    
    # Rename label column
    train_dataset = train_dataset.rename_column("label", "labels")
    val_dataset = val_dataset.rename_column("label", "labels")
    
    return train_dataset, val_dataset

In [3]:
# ============================================================================
# 3. TRAINING ENGINE: HF Trainer handles training, validation, and logging
# ============================================================================

# Note: The Trainer class from transformers handles:
# - Training loop with gradient accumulation
# - Validation every N steps (eval_steps=200)
# - Best model selection based on macro_f1
# - Automatic W&B logging if enabled
# - Checkpointing and model saving

In [ ]:
# ============================================================================
# 2. MODEL LAYER: Load pretrained CodeBERT and tokenizer
# ============================================================================

def load_model_and_tokenizer(model_name: str, num_labels: int = 2):
    """Load pretrained model and tokenizer from Hugging Face."""
    logger.info(f"Loading tokenizer from: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    logger.info(f"Loading model from: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="single_label_classification",
    )
    
    logger.info(f"Model loaded successfully")
    logger.info(f"Model config - num_labels: {model.config.num_labels}, hidden_size: {model.config.hidden_size}")
    
    return model, tokenizer


def freeze_base_model(model):
    """
    Freeze all base model parameters, keeping only classifier head trainable.
    
    For sequence classification models (e.g., CodeBERT), this freezes the 
    RoBERTa/encoder layers and unfreezes the classification head.
    
    Args:
        model: The pretrained model to freeze
    """
    # Freeze all encoder/base model parameters
    for name, param in model.named_parameters():
        if "classifier" not in name and "cls" not in name.lower():
            param.requires_grad = False
            
    # Ensure classifier head is trainable
    for name, param in model.named_parameters():
        if "classifier" in name or "cls" in name.lower():
            param.requires_grad = True
    
    logger.info("Base model frozen - only classifier head is trainable")


In [ ]:
# ============================================================================
# 2.5 LOSS FUNCTIONS: Custom loss functions following HF Trainer pattern
# ============================================================================

def focal_loss_compute(
    outputs,
    labels,
    num_items_in_batch=None,
    alpha: float = 1.0,
    gamma: float = 2.0,
):
    """
    Focal Loss computation following HF Trainer's compute_loss signature.
    
    Args:
        outputs: Model outputs containing logits
        labels: Ground truth labels
        num_items_in_batch: Number of items in batch (for compatibility)
        alpha: Focal loss alpha parameter (focus weight)
        gamma: Focal loss gamma parameter (focusing parameter)
    
    Returns:
        Focal loss tensor
    """
    logits = outputs.logits
    ce_loss = torch.nn.functional.cross_entropy(logits, labels, reduction="none")
    pt = torch.exp(-ce_loss)
    focal_loss = alpha * (1.0 - pt) ** gamma * ce_loss
    return focal_loss.mean()


def r_drop_loss_compute(
    outputs,
    labels,
    num_items_in_batch=None,
    alpha: float = 4.0,
):
    """
    R-Drop Loss computation following HF Trainer's compute_loss signature.
    Uses the model's built-in loss and adds KL divergence regularization.
    
    Args:
        outputs: Model outputs containing logits
        labels: Ground truth labels
        num_items_in_batch: Number of items in batch (for compatibility)
        alpha: Weight for KL divergence term (in bits, typically 4.0)
    
    Returns:
        R-Drop regularized loss tensor
    
    Note:
        For proper R-Drop implementation with two forward passes,
        use r_drop_with_dual_forward in the compute_loss of custom Trainer.
    """
    logits = outputs.logits
    ce_loss = torch.nn.functional.cross_entropy(logits, labels, reduction="mean")
    
    # Standard R-Drop with logits from single forward pass
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    probs = torch.nn.functional.softmax(logits, dim=-1)
    
    # KL divergence (approximation with single forward pass)
    kl_loss = torch.sum(probs * (log_probs - torch.nn.functional.log_softmax(logits, dim=-1)), dim=-1).mean()
    
    total_loss = ce_loss + alpha * kl_loss
    return total_loss


def infonce_loss_compute(
    outputs,
    labels,
    num_items_in_batch=None,
    temperature: float = 0.07,
    infonce_weight: float = 0.5,
):
    """
    InfoNCE Loss (Contrastive Loss) combined with Cross-Entropy following HF Trainer's compute_loss signature.
    
    Computes contrastive loss on hidden representations combined with classification loss.
    InfoNCE brings representations of same-label samples closer in embedding space.
    
    Args:
        outputs: Model outputs containing logits and hidden_state
        labels: Ground truth labels
        num_items_in_batch: Number of items in batch (for compatibility)
        temperature: Temperature parameter for softmax in contrastive learning (default: 0.07)
        infonce_weight: Weight for contrastive loss (total loss = CE + weight * InfoNCE)
    
    Returns:
        Combined loss tensor: CE loss + infonce_weight * InfoNCE loss
    
    Reference:
        Oord et al., "Representation Learning with Contrastive Predictive Coding"
    """
    logits = outputs.logits
    
    # Get hidden states (from last layer or use logits as representations)
    if hasattr(outputs, "hidden_states") and outputs.hidden_states is not None:
        # Use CLS token representation from last hidden layer
        hidden_states = outputs.hidden_states[-1]  # [batch_size, seq_len, hidden_dim]
        representations = hidden_states[:, 0, :]   # [batch_size, hidden_dim] - CLS token
    else:
        # Fallback: use logits as representations
        representations = logits
    
    # Normalize representations for cosine similarity
    representations = torch.nn.functional.normalize(representations, dim=-1)
    
    # Compute similarity matrix: [batch_size, batch_size]
    similarity_matrix = torch.matmul(representations, representations.t()) / temperature
    
    # Create positive/negative mask based on labels
    labels_expanded = labels.unsqueeze(1)  # [batch_size, 1]
    label_matches = (labels_expanded == labels_expanded.t()).float()  # [batch_size, batch_size]
    
    # Remove self-similarity from positive pairs
    mask = torch.eye(labels.size(0), device=labels.device, dtype=torch.bool)
    label_matches = label_matches.masked_fill(mask, 0)
    
    # Compute InfoNCE loss using label-based pairs
    exp_sim = torch.exp(similarity_matrix)
    
    # Sum of exponentials for the entire batch
    sum_exp_sim = exp_sim.sum(dim=1, keepdim=True)
    
    # For each sample, compute loss based on label matches
    batch_size = labels.size(0)
    infonce_loss = 0.0
    
    for i in range(batch_size):
        pos_mask = label_matches[i] > 0  # Positive pairs for sample i
        
        if pos_mask.sum() > 0:
            # At least one positive pair exists
            pos_sim = similarity_matrix[i][pos_mask].sum()
            neg_log_prob = -torch.log(exp_sim[i].sum() / sum_exp_sim[i].squeeze() + 1e-8)
            infonce_loss += neg_log_prob
        else:
            # No positive pairs for this sample, use cross-entropy as proxy
            pass
    
    infonce_loss = infonce_loss / batch_size
    
    # Combine CE loss with InfoNCE loss
    ce_loss = torch.nn.functional.cross_entropy(logits, labels)
    total_loss = ce_loss + infonce_weight * infonce_loss
    
    return total_loss

In [ ]:
def train_pipeline(
    model_name: str = "microsoft/codebert-base",
    output_dir: str = "./taskA-codebert-base",
    num_epochs: int = 3,
    batch_size: int = 32,
    learning_rate: float = 2e-5,
    max_length: int = 512,
    num_labels: int = 2,
    use_wandb: bool = False,
    freeze_base: bool = False,
    loss_type: str = "ce",
    focal_alpha: float = 1.0,
    focal_gamma: float = 2.0,
    r_drop_alpha: float = 4.0,
    infonce_temperature: float = 0.07,
    infonce_weight: float = 0.5,
):
    """
    Complete training pipeline using Hugging Face Trainer with custom loss support.
    
    Args:
        loss_type: Type of loss ("ce", "focal", "r-drop", "infonce")
        focal_alpha: Alpha parameter for Focal Loss
        focal_gamma: Gamma parameter for Focal Loss
        r_drop_alpha: Alpha parameter for R-Drop
        infonce_temperature: Temperature for InfoNCE contrastive loss
        infonce_weight: Weight of InfoNCE loss (total = CE + weight * InfoNCE)
    """
    from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(output_dir, exist_ok=True)
    
    # Setup file logging to output directory
    log_file = os.path.join(output_dir, "training.log")
    file_handler = logging.FileHandler(log_file, mode="w")
    file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(name)s - %(message)s"))
    logger.addHandler(file_handler)
    logger.info(f"Logging to file: {log_file}")
    
    # Load model and tokenizer
    model, tokenizer = load_model_and_tokenizer(model_name, num_labels)
    model = model.to(device)
    logger.info(f"Model on device: {device}")
    
    # For InfoNCE loss, we need hidden states output
    if loss_type == "infonce":
        model.config.output_hidden_states = True
        logger.info("Enabled hidden states output for InfoNCE loss")
    
    # Freeze base model if requested
    if freeze_base:
        logger.info("Freezing base model weights, training classifier head only...")
        freeze_base_model(model)
    
    # Log model architecture
    logger.info("\n" + "=" * 80)
    logger.info("MODEL ARCHITECTURE")
    logger.info("=" * 80)
    model_architecture = str(model)
    logger.info(model_architecture)
    logger.info("=" * 80 + "\n")
    
    # Count total and trainable parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params
    
    logger.info(f"Total parameters: {total_params:,}")
    logger.info(f"Trainable parameters: {trainable_params:,}")
    logger.info(f"Frozen parameters: {frozen_params:,}")
    logger.info(f"Trainable ratio: {100 * trainable_params / total_params:.2f}%\n")
    
    if freeze_base:
        logger.info("TRAINING MODE: Classifier head only (base model frozen)")
    else:
        logger.info("TRAINING MODE: Full model fine-tuning")
    logger.info("")
    
    # Load and prepare datasets
    train_dataset, val_dataset = load_datasets(tokenizer, max_length)
    
    # Setup weights and biases logging if enabled
    if use_wandb:
        try:
            import wandb
            os.environ["WANDB_MODE"] = "offline"
            logger.info("W&B set to offline mode for notebook stability")
        except Exception as e:
            logger.warning(f"Could not setup W&B: {e}. Continuing without W&B.")
            use_wandb = False
    
    # Calculate training parameters
    steps_per_epoch = max(1, len(train_dataset) // batch_size)
    total_steps = num_epochs * steps_per_epoch
    warmup_steps = max(1, int(total_steps * 0.1))
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        weight_decay=0.01,
        learning_rate=learning_rate,
        warmup_steps=warmup_steps,
        logging_strategy="steps",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to=["wandb"] if use_wandb else [],
        run_name="codebert-base-task-a" if use_wandb else None,
        gradient_checkpointing=True,
        dataloader_pin_memory=True if torch.cuda.is_available() else False,
        seed=42,
    )
    
    # Create trainer with custom loss function
    data_collator = DataCollatorWithPadding(tokenizer)
    
    # Prepare compute_loss_func based on loss_type
    compute_loss_func = None
    if loss_type == "focal":
        logger.info(f"Using Focal Loss with alpha={focal_alpha}, gamma={focal_gamma}")
        compute_loss_func = lambda outputs, labels, num_items_in_batch=None: focal_loss_compute(
            outputs, labels, num_items_in_batch, alpha=focal_alpha, gamma=focal_gamma
        )
    elif loss_type == "r-drop":
        logger.info(f"Using R-Drop Loss with alpha={r_drop_alpha}")
        compute_loss_func = lambda outputs, labels, num_items_in_batch=None: r_drop_loss_compute(
            outputs, labels, num_items_in_batch, alpha=r_drop_alpha
        )
    elif loss_type == "infonce":
        logger.info(f"Using InfoNCE Loss with temperature={infonce_temperature}, weight={infonce_weight}")
        compute_loss_func = lambda outputs, labels, num_items_in_batch=None: infonce_loss_compute(
            outputs, labels, num_items_in_batch, temperature=infonce_temperature, infonce_weight=infonce_weight
        )
    elif loss_type == "ce":
        logger.info("Using Cross-Entropy Loss (default)")
        compute_loss_func = None
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}. Choose from ['ce', 'focal', 'r-drop', 'infonce']")
    
    # Create custom Trainer with compute_loss_func
    class CustomTrainer(Trainer):
        def __init__(self, *args, compute_loss_func=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.compute_loss_func = compute_loss_func
        
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            """Override compute_loss to use custom loss function."""
            if self.compute_loss_func is not None and "labels" in inputs:
                labels = inputs.pop("labels")
            else:
                labels = None
            
            outputs = model(**inputs)
            
            if labels is not None and self.compute_loss_func is not None:
                loss = self.compute_loss_func(outputs, labels, num_items_in_batch=num_items_in_batch)
            elif labels is not None:
                # Default: use model's built-in loss
                loss = torch.nn.functional.cross_entropy(outputs.logits, labels)
            else:
                loss = outputs.loss if hasattr(outputs, "loss") else outputs[0]
            
            return (loss, outputs) if return_outputs else loss
    
    trainer = CustomTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fn,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        compute_loss_func=compute_loss_func,
    )
    
    # Log training configuration
    logger.info("\n" + "=" * 80)
    logger.info("TRAINING CONFIGURATION")
    logger.info("=" * 80)
    logger.info(f"Model: {model_name}")
    logger.info(f"Output directory: {output_dir}")
    logger.info(f"Epochs: {num_epochs}")
    logger.info(f"Batch size: {batch_size}")
    logger.info(f"Learning rate: {learning_rate}")
    logger.info(f"Max sequence length: {max_length}")
    logger.info(f"Warmup steps: {warmup_steps} / {total_steps}")
    logger.info(f"Evaluation strategy: every 200 steps")
    logger.info(f"Gradient checkpointing: True")
    logger.info(f"Freeze base model: {freeze_base}")
    logger.info(f"Loss function: {loss_type}")
    if loss_type == "focal":
        logger.info(f"  - Focal alpha: {focal_alpha}")
        logger.info(f"  - Focal gamma: {focal_gamma}")
    if loss_type == "r-drop":
        logger.info(f"  - R-Drop alpha: {r_drop_alpha}")
    if loss_type == "infonce":
        logger.info(f"  - InfoNCE temperature: {infonce_temperature}")
        logger.info(f"  - InfoNCE weight: {infonce_weight}")
    logger.info(f"W&B logging: {use_wandb}")
    logger.info("=" * 80 + "\n")
    
    # Train
    logger.info("Starting training...")
    trainer.train()
    
    logger.info(f"\n🎯 Training complete!")
    logger.info(f"Best model saved to: {os.path.join(output_dir, 'best_model')}")
    
    return trainer, model, tokenizer

In [ ]:
# Option 1: Cross-Entropy (default)
# trainer, model, tokenizer = train_pipeline(loss_type="ce")

# Option 2: Focal Loss (for imbalanced classes)
# trainer, model, tokenizer = train_pipeline(loss_type="focal", focal_alpha=1.0, focal_gamma=2.0)

# Option 3: R-Drop (regularization through sample dropout)
# trainer, model, tokenizer = train_pipeline(loss_type="r-drop", r_drop_alpha=4.0)

# Option 4: InfoNCE Contrastive Loss (representation-based learning)
# trainer, model, tokenizer = train_pipeline(
#     loss_type="infonce",
#     infonce_temperature=0.07,
#     infonce_weight=0.5,
# )

# Uncomment one of the above to run training with that loss function
trainer, model, tokenizer = train_pipeline()

04/15/2026 01:15:51 - INFO - __main__ - Logging to file: ./taskA-codebert-base\training.log
04/15/2026 01:15:51 - INFO - __main__ - Loading tokenizer from: microsoft/codebert-base
04/15/2026 01:15:51 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/15/2026 01:15:52 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/config.json "HTTP/1.1 200 OK"
04/15/2026 01:15:52 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
04/15/2026 01:15:52 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/tokenizer_config.json "HTTP/1.1 200 OK"
04/15/2026 01:15:52 - INFO - httpx - HTTP Request: GET https://huggingface.co/

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
04/15/2026 01:15:54 - INFO - __main__ - Model loaded successfully
04/15/2026 01:15:54 - INFO - __main__ - Model config - num_labels: 2, hidden_size: 768
04/15/2026 01:15:54 - INFO - __main__ - Model on device: cpu
04/15/2026 01:15:54 - INFO - __main__ - 
04/15/2026 01:15:54 - INFO - __main__ - MODEL ARCHITECTURE
04/15/2026 01:15:54

Step,Training Loss,Validation Loss


04/15/2026 01:15:51 - INFO - __main__ - Logging to file: ./taskA-codebert-base\training.log
04/15/2026 01:15:51 - INFO - __main__ - Loading tokenizer from: microsoft/codebert-base
04/15/2026 01:15:51 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/15/2026 01:15:52 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/config.json "HTTP/1.1 200 OK"
04/15/2026 01:15:52 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
04/15/2026 01:15:52 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/tokenizer_config.json "HTTP/1.1 200 OK"
04/15/2026 01:15:52 - INFO - httpx - HTTP Request: GET https://huggingface.co/

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
04/15/2026 01:15:54 - INFO - __main__ - Model loaded successfully
04/15/2026 01:15:54 - INFO - __main__ - Model config - num_labels: 2, hidden_size: 768
04/15/2026 01:15:54 - INFO - __main__ - Model on device: cpu
04/15/2026 01:15:54 - INFO - __main__ - 
04/15/2026 01:15:54 - INFO - __main__ - MODEL ARCHITECTURE
04/15/2026 01:15:54

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [15]:
# ============================================================================
# 5. INFERENCE: Generate predictions on test set
# ============================================================================
def inference_pipeline(
    trainer,
    tokenizer,
    output_csv: str = "submission.csv",
    batch_size: int = 32,
    max_length: int = 512,
):
    """Generate predictions on test set using trained model."""
    logger.info("Loading test dataset...")
    test_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="test")
    
    # Tokenize test set using same tokenizer
    logger.info("Tokenizing test dataset...")
    test_dataset = test_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in test_dataset.column_names],
        desc="Tokenizing test",
    )
    
    # Generate predictions using the trained model
    logger.info("Generating predictions on test set...")
    predictions = trainer.predict(test_dataset)
    logits = predictions.predictions
    pred_labels = logits.argmax(axis=-1)
    
    # Write to CSV
    logger.info(f"Writing predictions to {output_csv}...")
    with open(output_csv, "w") as f:
        f.write("id,label\n")
        for i, pred in enumerate(pred_labels):
            f.write(f"{i},{pred}\n")
    
    logger.info(f"✅ Predictions saved to {output_csv}")
    logger.info(f"Total predictions: {len(pred_labels)}")
    logger.info(f"Class distribution: {[(pred_labels == i).sum() for i in range(2)]}")
    
    return pred_labels


# Run inference using the trained model
logger.info("\n" + "=" * 80)
logger.info("INFERENCE")
logger.info("=" * 80 + "\n")

predictions = inference_pipeline(
    trainer=trainer,
    tokenizer=tokenizer,
    output_csv="submission.csv",
    batch_size=32,
    max_length=512,
)

04/15/2026 01:38:37 - INFO - __main__ - 
04/15/2026 01:38:37 - INFO - __main__ - INFERENCE
04/15/2026 01:38:37 - INFO - __main__ - ================================================================================



04/15/2026 01:38:37 - INFO - __main__ - 
04/15/2026 01:38:37 - INFO - __main__ - INFERENCE
04/15/2026 01:38:37 - INFO - __main__ - ================================================================================



NameError: name 'trainer' is not defined

In [ ]:
# ============================================================================
# 6. (OPTIONAL) Upload trained model to Hugging Face Hub
# ============================================================================

def upload_to_hub(
    trainer,
    repo_id: str = "dzungpham/taskA-codebert-base",
    commit_message: str = "Upload fine-tuned CodeBERT model"
):
    """Upload the trained model to Hugging Face Hub."""
    try:
        logger.info(f"Uploading model to {repo_id}...")
        trainer.push_to_hub(
            repo_id=repo_id,
            commit_message=commit_message
        )
        logger.info(f"✅ Model uploaded to: https://huggingface.co/{repo_id}")
    except Exception as e:
        logger.warning(f"Could not upload to Hub: {e}")


# Uncomment the line below to upload the model to HF Hub
# upload_to_hub(trainer, repo_id="your-username/taskA-codebert-base")